# Try the FunctionGemma voice-command model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peterblazejewicz/functiongemma-270m-it-gemma-teacher/blob/main/quick-start.ipynb)

This notebook runs [`blazejewicz/functiongemma-270m-it-gemma-teacher`](https://huggingface.co/blazejewicz/functiongemma-270m-it-gemma-teacher) with Hugging Face Transformers. You type
what a person would say to a screen-free voice appliance, in English or Polish, and the model answers with one
function call from a small declared set, or with no call when the words are not a command.

**Before you start:**

1. Open the [model page](https://huggingface.co/blazejewicz/functiongemma-270m-it-gemma-teacher), read the terms, and click **Agree and get access**.
   The files are gated.
2. Make a Hugging Face [access token](https://huggingface.co/settings/tokens) of the type **Read**.
3. In Colab, click the key icon (**Secrets**) on the left, add a secret with the name `HF_TOKEN` and your token as the
   value, and give this notebook access to it. Without the secret, the notebook asks for the token.
4. The model has 270 million parameters: the free CPU runtime is enough, and no GPU is necessary.
5. Select **Runtime > Run all**.

The notebook also runs in Jupyter on your computer.

In [ ]:
# @title Settings
MODEL = "blazejewicz/functiongemma-270m-it-gemma-teacher"  # @param {type:"string"}
REVISION = "main"  # @param {type:"string"}

In [ ]:
# @title 1. Load the model and its declarations
%pip install -q "transformers>=4.57.3" huggingface_hub

import json
import os
import re
import time
from getpass import getpass
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from transformers import AutoModelForCausalLM, AutoTokenizer


def colab_secret(name):
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


local = Path(MODEL).is_dir()
token = None
if not local:
    token = (os.environ.get("HF_TOKEN") or colab_secret("HF_TOKEN") or getpass("Hugging Face token: ")).strip()
    if not token:
        raise SystemExit("A Hugging Face token of the type Read is necessary. See step 2 above.")
try:
    tools_path = Path(MODEL) / "tools.json" if local else hf_hub_download(MODEL, "tools.json", revision=REVISION,
                                                                         token=token)
    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, token=token)
    model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, token=token, dtype=torch.float32)
except GatedRepoError:
    raise SystemExit(f"Open https://huggingface.co/{MODEL}, click 'Agree and get access', and run this cell again.")
except HfHubHTTPError as error:
    raise SystemExit(f"The Hub refused the download: {error}. Make sure that the token is correct and can read.")
model.eval()
TOOLS = json.loads(Path(tools_path).read_text(encoding="utf-8"))
import transformers
print(f"transformers {transformers.__version__}, torch {torch.__version__}")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, {len(TOOLS)} declared functions:")
for tool in TOOLS:
    description = re.sub(r"^[A-Z-]+\. ", "", tool["function"]["description"]).split(". Synonyms")[0]
    print(" ", tool["function"]["name"], "-", description)

In [ ]:
# @title 2. The function that asks the model
END_OF_CALL = "<end_function_call>"
CALL = re.compile(r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", re.DOTALL)
ARGUMENT = re.compile(r"(\w+):<escape>(.*?)<escape>", re.DOTALL)
end_ids = [tokenizer.convert_tokens_to_ids(END_OF_CALL), tokenizer.eos_token_id]


def propose(utterance):
    """One user turn with the declarations; greedy decoding up to the end of the first call."""
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": utterance}], tools=TOOLS,
                                           add_generation_prompt=True, return_dict=True, return_tensors="pt")
    started = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=64, do_sample=False, eos_token_id=end_ids)
    seconds = time.perf_counter() - started
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False)
    match = CALL.search(text)
    if match is None:
        return None, text, seconds
    return (match.group(1), dict(ARGUMENT.findall(match.group(2)))), text, seconds


def show(utterance):
    call, text, seconds = propose(utterance)
    if call is None:
        print(f"{utterance!r}\n  -> no command ({seconds:.2f} s)")
    else:
        name, arguments = call
        print(f"{utterance!r}\n  -> {name}({json.dumps(arguments, ensure_ascii=False)}) ({seconds:.2f} s)")

In [ ]:
# @title 3. Say something to the appliance
utterance = "Ustaw długość wyjaśnień na szczegółową"  # @param {type:"string"}
show(utterance)

Change the words and run the cell again, or run the next cell for a set of examples. A command gives one call; a
question for the teacher, small talk, a negation ("don't change the level") and a request that the appliance does not
take by voice (confirm, cancel, leave the menu, stop) give **no command**.

In [ ]:
# @title 4. Examples in English and Polish
for utterance in [
    "set the level to intermediate",
    "Mów wolniej, proszę.",
    "What is the explanation length now?",
    "Jakie są opcje języka menu?",
    "next card",
    "Powtórz, proszę.",
    "make the explanations brief... no, wait, detailed",
    "How do I say thank you in Japanese?",
    "Co znaczy mizu?",
    "Don't change the teacher language.",
    "Tak, potwierdzam.",
]:
    show(utterance)